# Comm throughput spike: how fast is binary data over the widget comm?

This notebook is a **standalone measurement**, not part of the dashboard. It
answers one question the dashboard's design leans on but has never
measured: how fast is a binary buffer over the ipywidgets comm in this
kernel (xeus-python, WASM, running under Voici/JupyterLite), and does a
single large buffer even survive the trip?

**This must be run in the browser.** A native `xpython` or CPython kernel
will report numbers that mean nothing here -- the whole point is the wasm
build's comm/serialization path, which has no native equivalent. Run every
cell below, in order, inside the deployed site (or `pixi run jupyter lab`
against the JupyterLite build), not natively.

Two things this settles:

1. **Does a single 4 MB buffer round-trip at all** under this kernel? If it
   silently truncates or the kernel wedges, that alone rules out ever
   raising `CHUNK_BYTES` past whatever does survive.
2. **MB/s in each direction**, which is what decides whether the current
   1 MiB chunking (`CHUNK_BYTES = 1 << 20` in `content/photom_dashboard.py`)
   is costing anything. If per-byte cost at 4 MiB is about the same as at
   1 MiB, chunking is free and 1 MiB should stay; if 4 MiB is dramatically
   cheaper per byte, it is worth raising `CHUNK_BYTES` (at the cost of more
   peak memory per frame).

The output of this notebook is a number (or two): write the measured
MB/s figures into `docs/dashboard.md` and `PROGRESS.md`, and update
`CHUNK_BYTES` if the data says to.

In [ ]:
# A tiny, self-contained anywidget for round-tripping binary buffers.
#
# Deliberately NOT `from dropzone import ...`: this notebook has to keep
# working as a throughput probe even if the dashboard's real widgets
# change shape, so its protocol is defined inline and from scratch.
#
# Wire protocol (mirrors the shape of the real dashboard's comm, but is
# otherwise unrelated to it):
#
#   JS -> kernel   {type:'ready'}   (render handshake: safe to send now)
#   kernel -> JS   {type:'down', id, nbytes, crc32} + 1 buffer
#   JS -> kernel   {type:'down_arrived', id, nbytes, got}   (pre-CRC: timing)
#   JS -> kernel   {type:'down_done', id, nbytes, ok}       (post-CRC: verdict)
#   kernel -> JS   {type:'up_request', id, nbytes}
#   JS -> kernel   {type:'up', id, nbytes, crc32} + 1 buffer
#
# `nbytes` in every message is the REQUESTED size, not necessarily the
# size actually received. `crc32` is a checksum of the buffer the SENDER
# put on the wire; the receiver recomputes it over what actually arrived
# and folds "length matches AND checksum matches" into `ok` (down: JS
# computes it and reports the verdict back in `down_done`; up: the kernel
# computes it in `_on_msg`, in the driver cell below). That is what turns a
# silently truncated OR corrupted buffer into a visible failure instead of
# a suspiciously fast number -- length alone would miss corruption that
# happens to preserve size.
#
# `down_arrived` exists so the CRC pass stays OUT of the measured window:
# Python's round-trip clock stops on its receipt, while the O(size) JS-side
# crc32 runs after it is sent and reports separately in `down_done`. With a
# single post-CRC reply the checksum sat inside the timing bracket, and
# asymmetrically so between the 1 MiB and 4 MiB cases -- the very
# comparison this notebook exists to make.
import anywidget

_ESM = r"""
function render({ model, el }) {
  el.innerHTML = '';

  // A visible log matters here more than usual: if the Python-side timer
  // never fires (comm wedged, reply dropped), this is the only evidence
  // that anything happened at all.
  const log = document.createElement('pre');
  log.style.fontSize = '0.85em';
  log.style.maxHeight = '220px';
  log.style.overflow = 'auto';
  log.style.background = 'var(--jp-layout-color2, #f5f5f5)';
  log.style.padding = '0.5em';
  el.appendChild(log);

  function line(msg) {
    log.textContent += msg + '\n';
    log.scrollTop = log.scrollHeight;
  }
  line('spike_comm widget ready.');

  // Non-zero, non-constant fill: a buffer that arrives all-zero (e.g. a
  // dropped/short comm write silently zero-filled somewhere) must not be
  // mistaken for a correctly-sent one just because the length matches.
  function fillPattern(n) {
    const buf = new Uint8Array(n);
    for (let i = 0; i < n; i++) buf[i] = (i * 2654435761) & 0xff;
    return buf;
  }

  // Standard CRC-32 (the CRC-32/ISO-HDLC variant zlib.crc32 implements),
  // so the JS and Python sides can agree on a single checksum without
  // pulling in a library. Table built lazily, once, on first use.
  let _crcTable = null;
  function crc32(bytes) {
    if (!_crcTable) {
      _crcTable = new Uint32Array(256);
      for (let n = 0; n < 256; n++) {
        let c = n;
        for (let k = 0; k < 8; k++) {
          c = (c & 1) ? (0xedb88320 ^ (c >>> 1)) : (c >>> 1);
        }
        _crcTable[n] = c >>> 0;
      }
    }
    let crc = 0xffffffff;
    for (let i = 0; i < bytes.length; i++) {
      crc = _crcTable[(crc ^ bytes[i]) & 0xff] ^ (crc >>> 8);
    }
    return (crc ^ 0xffffffff) >>> 0;
  }

  function onCustomMessage(msg, buffers) {
    if (!msg) return;

    if (msg.type === 'down') {
      const t = performance.now();
      const raw = buffers && buffers[0];
      // A Python bytes/memoryview arrives here as a DataView, not a
      // Uint8Array -- byteLength works on both, .length does not.
      const got = raw ? raw.byteLength : 0;
      // Timing reply FIRST, before the crc32 pass: Python stops its
      // round-trip clock on this message. The CRC is O(size), so leaving
      // it ahead of the only reply (the old shape) put it inside the
      // measured window -- and by a different amount at 1 MiB vs 4 MiB.
      model.send({ type: 'down_arrived', id: msg.id, nbytes: msg.nbytes, got });
      // The verdict still folds content into `ok` (catching corruption
      // that preserves length); it just travels in a second, untimed
      // message now.
      const gotBytes = raw
        ? new Uint8Array(raw.buffer, raw.byteOffset, raw.byteLength)
        : new Uint8Array(0);
      const matches = crc32(gotBytes) === msg.crc32;
      const ok = got === msg.nbytes && matches;
      line(`down id=${msg.id}: got ${got}/${msg.nbytes} bytes, ` +
           `content matches=${matches}, ok=${ok}, t=${t.toFixed(1)}ms`);
      model.send({ type: 'down_done', id: msg.id, nbytes: msg.nbytes, ok });
      return;
    }

    if (msg.type === 'up_request') {
      const bytes = fillPattern(msg.nbytes);
      // Include a checksum of what's actually being sent so the kernel
      // can verify content, not just the byte count it already gets from
      // len(buffers[0]).
      const crc = crc32(bytes);
      line(`up_request id=${msg.id}: sending ${msg.nbytes} bytes`);
      model.send(
        { type: 'up', id: msg.id, nbytes: msg.nbytes, crc32: crc },
        null,
        [bytes.buffer]
      );
      return;
    }
  }
  model.on('msg:custom', onCustomMessage);

  // Handshake, sent strictly AFTER the handler registration above. This
  // view only starts listening once render() runs -- after the
  // display_data is processed, this module is dynamically imported, and
  // the view attaches -- and a custom message that reaches the model
  // before then is dropped with no error anywhere. The kernel therefore
  // holds its first transfer until this message arrives (see the driver
  // cell) instead of firing into that gap.
  model.send({ type: 'ready' });

  return () => model.off('msg:custom', onCustomMessage);
}

export default { render };
"""


class CommSpike(anywidget.AnyWidget):
    """Round-trips binary buffers of a chosen size, for timing only."""

    _esm = _ESM

In [ ]:
# The measurement driver.
#
# THE ONE CONFUSING THING ABOUT THIS NOTEBOOK: comm messages are only
# delivered to Python while the kernel is IDLE (between cells, not during
# one). A cell that sends a request and then sits in a loop waiting for the
# reply would block forever -- the reply is queued but never handled, because
# handling it requires the kernel to go idle, which it never does while
# stuck in that loop. This is the same constraint `photom_dashboard.py`'s
# docstring calls out for the real dashboard: "the kernel only receives
# comm messages while it is idle, so there is no loop".
#
# The fix here is the same shape as the dashboard's: this cell only SENDS
# and returns immediately. `_on_msg` (registered once, below) is invoked by
# the kernel between cells as replies trickle in, and records arrival times
# into RESULTS. A LATER cell (run after waiting a few seconds, possibly more
# than once) reads whatever has accumulated in RESULTS so far.
#
# A SECOND trap, the mirror image of the first: the JS side only starts
# listening once its view has rendered (the msg:custom handler is
# registered inside render(), which runs asynchronously well after
# `display(widget)` returns -- cold cache, slower still), and a custom
# message sent before that is dropped with no error on either side. This
# cell used to end with a direct `_pump()` call, racing the first 'down'
# against that render; a fresh incognito window reliably lost the race and
# every row wedged at "(no reply yet)" (the 2026-08-22 run). So nothing is
# sent here at all: JS sends a 'ready' handshake from render(), after its
# handler is registered, and `_on_msg` starts the queue on its arrival.
import os
import time
import zlib

from IPython.display import display

SIZES = [
    64 * 1024,     # 64 KiB
    256 * 1024,    # 256 KiB
    1 << 20,       # 1 MiB -- current CHUNK_BYTES
    4 << 20,       # 4 MiB -- the size this notebook exists to check
    4_150_000,     # a real Seestar frame's on-disk byte count
]

widget = CommSpike()
display(widget)

RESULTS = {}    # (direction, nbytes) -> {"seconds": float, "ok": bool|None}
_SENT_AT = {}   # (direction, id) -> time.monotonic() when the request left
_ARRIVED = {}   # id -> nbytes, for downs timed but not yet CRC-verdicted
_STARTED = [False]   # set by the first 'ready'; read by the results cell
_next_id = [0]


def _new_id():
    _next_id[0] += 1
    return _next_id[0]


def _on_msg(_widget, content, buffers):
    # Timed here, on receipt in Python -- this is a ROUND TRIP (Python send
    # to Python receive), not a one-way number. It includes JS-side dispatch
    # and, for 'down', the small 'down_done' reply travelling back; for
    # 'up', the small 'up_request' travelling out. That makes every figure
    # below a lower bound on one-way throughput, not a pure serialization
    # measurement -- honest, but not the tightest possible number.
    now = time.monotonic()
    kind = content.get("type")

    if kind == "ready":
        # The view has rendered and its handler is registered -- only now
        # is it safe to put the first transfer on the wire. Guarded: a
        # re-rendered view (output re-attached, notebook reopened) sends
        # 'ready' again and must not restart the queue mid-run.
        if not _STARTED[0]:
            _STARTED[0] = True
            _pump()
        return

    if kind == "down_arrived":
        key = ("down", content["id"])
        t0 = _SENT_AT.pop(key, None)
        if t0 is None:
            return  # stray/duplicate reply; ignore rather than crash a cell
        # The clock stops HERE, before JS runs its O(size) crc32 over the
        # payload; the verdict follows in `down_done` and fills in `ok`.
        RESULTS[("down", content["nbytes"])] = {
            "seconds": now - t0,
            "ok": None,
        }
        _ARRIVED[content["id"]] = content["nbytes"]
        return  # not _pump(): the CRC and down_done are still in flight

    if kind == "down_done":
        nbytes = _ARRIVED.pop(content["id"], None)
        if nbytes is None:
            return  # stray/duplicate verdict
        # `ok` folds in JS's own crc32 check (see the widget cell) --
        # content, not just length, verified on that end.
        RESULTS[("down", nbytes)]["ok"] = bool(content.get("ok"))

    elif kind == "up":
        key = ("up", content["id"])
        t0 = _SENT_AT.pop(key, None)
        if t0 is None:
            return
        got = len(buffers[0]) if buffers else 0
        # `now` was already captured above, before this runs, so a crc32
        # pass over a <=4 MiB buffer (a few ms at most) can't skew the
        # round-trip time being recorded below. Checking the checksum the
        # JS sender computed over its own fillPattern output is what makes
        # this a content check rather than just a length check.
        content_ok = bool(buffers) and zlib.crc32(buffers[0]) == content.get(
            "crc32"
        )
        RESULTS[("up", content["nbytes"])] = {
            "seconds": now - t0,
            "ok": got == content["nbytes"] and content_ok,
        }
    else:
        return

    _pump()  # one transfer in flight at a time -- see below


widget.on_msg(_on_msg)


def send_down(nbytes):
    # os.urandom, not zeros: a compressible/constant buffer could let some
    # layer of the stack cheat, and this must measure the real cost of
    # opaque FITS bytes.
    payload = os.urandom(nbytes)
    # Computed BEFORE the timer starts below, so it can't inflate the
    # round-trip time being measured -- and at a few ms for <=4 MiB it
    # would be negligible even if it landed inside the bracket. The JS
    # receiver recomputes this over what actually arrived and folds the
    # comparison into `ok`.
    crc = zlib.crc32(payload)
    msg_id = _new_id()
    _SENT_AT[("down", msg_id)] = time.monotonic()
    msg = {"type": "down", "id": msg_id, "nbytes": nbytes, "crc32": crc}
    widget.send(msg, buffers=[payload])


def send_up_request(nbytes):
    msg_id = _new_id()
    _SENT_AT[("up", msg_id)] = time.monotonic()
    widget.send({"type": "up_request", "id": msg_id, "nbytes": nbytes})


# Exactly ONE transfer is in flight at a time: the first is started by the
# 'ready' handshake, each later one by the reply to the previous. Firing
# all ten at once would be simpler, but every round trip after the first
# would then include the time spent draining ~9.5 MB of earlier payloads
# out of the comm -- which would make the large sizes look expensive purely
# because they are queued last, and the 1 MiB vs 4 MiB per-byte comparison
# is the entire point of this notebook.
_QUEUE = [(d, n) for n in SIZES for d in ("down", "up")]


def _pump():
    if not _QUEUE:
        return
    direction, nbytes = _QUEUE.pop(0)
    (send_down if direction == "down" else send_up_request)(nbytes)


print(f"Queued {len(_QUEUE)} transfers "
      f"({len(SIZES)} sizes x 2 directions), one at a time; the first")
print("starts when the widget above reports ready (its log's first line).")
print("Watch the widget's log for progress, then run the NEXT cell")
print("(re-run it if some rows still say 'no reply yet' -- give it a few")
print("seconds, especially for the 4 MiB case).")

In [ ]:
# Results table. Only replies that have actually arrived are in RESULTS --
# see the comment in the driver cell above for why this can't just block
# until everything is in. Re-run this cell (Cell -> Run) after a few
# seconds if any row still says "no reply yet".
DIRECTIONS = ("down", "up")

if not _STARTED[0]:
    # The 'ready' handshake never arrived, so no transfer has even been
    # sent: the widget's view has not rendered (or its ESM failed to
    # load). Waiting longer only helps if the page is still loading;
    # otherwise check the browser console and that the widget cell above
    # actually shows the log box with "spike_comm widget ready."
    print("Widget never reported ready -- NO transfers have started yet.")
    print("If the widget's log box is missing above, the view failed to")
    print("render; check the browser console.\n")

print(f"{'dir':5s} {'bytes':>10s} {'seconds':>9s} {'MB/s':>9s}  ok")
rows = []
for n in SIZES:
    for direction in DIRECTIONS:
        r = RESULTS.get((direction, n))
        if r is None:
            print(f"{direction:5s} {n:10d}   (no reply yet)")
            continue
        if r["ok"] is None:
            # The down was timed (down_arrived landed) but its CRC
            # verdict (down_done) has not -- distinct from "no reply
            # yet": if THIS state persists, the verdict message is being
            # lost, not the transfer itself.
            print(f"{direction:5s} {n:10d} {r['seconds']:9.3f}"
                  f"   (CRC verdict pending)")
            continue
        mbps = (n / 1e6) / r["seconds"] if r["seconds"] > 0 else float("inf")
        rows.append((direction, n, r["seconds"], mbps, r["ok"]))
        secs, ok = r["seconds"], r["ok"]
        print(f"{direction:5s} {n:10d} {secs:9.3f} {mbps:9.1f}  {ok}")

if len(rows) < len(DIRECTIONS) * len(SIZES):
    print("\nStill waiting on some replies -- re-run this cell in a moment.")
else:
    # Verdict, computed only once every size has answered. Both directions
    # of the 4 MiB case must have arrived AND verified intact -- length
    # AND checksum matching, per the crc32 check on each receiver -- for
    # it to count as "survives the trip".
    four = [r for r in rows if r[1] == 4 << 20]
    one = [r for r in rows if r[1] == 1 << 20]
    four_ok = all(r[4] for r in four)
    # Per-byte cost, averaged over both directions, at each size.
    cost_1m = sum(r[2] for r in one) / sum(r[1] for r in one)
    cost_4m = sum(r[2] for r in four) / sum(r[1] for r in four)
    ratio = cost_1m / cost_4m if cost_4m > 0 else float("nan")

    print(f"\n4 MiB round trip ok in both directions: {four_ok}")
    print(f"per-byte cost at 1 MiB vs 4 MiB: {ratio:.2f}x")
    if not four_ok:
        verdict = "4 MiB failed -- keep CHUNK_BYTES = 1 << 20 (or lower)."
    elif ratio < 1.2:
        verdict = (
            "4 MiB is not meaningfully cheaper per byte -- 1 MiB chunking "
            "is free; keep CHUNK_BYTES = 1 << 20."
        )
    else:
        verdict = (
            "4 MiB is meaningfully cheaper per byte -- raising CHUNK_BYTES "
            "would pay off; weigh that against the extra peak memory."
        )
    print(f"verdict: {verdict}")

## Acting on the numbers

Fill in the measured columns below (from the table the previous cell
printed), then apply the decision rules underneath. Record the final
numbers in `docs/dashboard.md` and `PROGRESS.md` -- this notebook is a
one-off measurement, not a place those numbers live long-term.

Measured 2026-08-22 (clean run: fresh build, fresh browser origin, `ready`
handshake in place; recorded in `docs/dashboard.md` §8):

| size | direction | seconds (measured) | MB/s (measured) | ok (measured) |
|---|---|---|---|---|
| 64 KiB | down | 0.007 | 10.1 | True |
| 64 KiB | up | 0.007 | 9.1 | True |
| 256 KiB | down | 0.006 | 44.4 | True |
| 256 KiB | up | 0.008 | 35.0 | True |
| 1 MiB | down | 0.006 | 184.0 | True |
| 1 MiB | up | 0.011 | 94.5 | True |
| 4 MiB | down | 0.018 | 234.3 | True |
| 4 MiB | up | 0.023 | 180.8 | True |
| 4,150,000 B | down | 0.019 | 220.7 | True |
| 4,150,000 B | up | 0.021 | 193.9 | True |

The verdict cell printed a per-byte ratio of 1.64x (above its mechanical
1.2x threshold), but see `docs/dashboard.md` §8: that ratio is fixed
per-message overhead amortizing, and in absolute terms comm is ~44 ms of a
~2 s frame, so the recommendation is to keep `CHUNK_BYTES = 1 << 20`.

**Decision rules:**

- **4 MiB fails, or its `ok` is ever False, or it is much slower per byte
  than 1 MiB:** keep `CHUNK_BYTES = 1 << 20` in `content/photom_dashboard.py`
  -- the current chunking is already close to free and there is nothing to
  gain from a riskier larger buffer.
- **Larger buffers are dramatically cheaper per byte (comfortably above the
  1.2x threshold the verdict cell uses):** raising `CHUNK_BYTES` is worth
  it. Note the memory cost when doing so -- peak kernel memory during an
  upload is one chunk plus whatever MEMFS already holds for that file, so a
  bigger chunk raises the floor directly.
- **Binary comm fails outright in one or both directions** (not just slow --
  wrong, truncated, or the kernel wedges): the fallback is base64 over the
  existing JSON `msg:custom` channel, at roughly 33% size overhead. That
  changes only the transport underneath `ChunkAssembler`/the JS chunk loop,
  not the chunking protocol itself.

Whatever the outcome, write the measured MB/s (both directions) and the
4 MiB pass/fail result into `docs/dashboard.md` (wire protocol section) and
into `PROGRESS.md`, so the next person does not have to re-run this spike.